# Tiny Aya: a readable Japanese/English evaluation pipeline

This notebook is the experiment's main interface. Read and run each section in order: **load data → generate or reuse answers → judge → compare → inspect → save**.

Start with `MODE = "saved"` to reanalyze the existing 28,800-answer experiment on CPU. It reads the original SQLite/JSONL files and writes corrected results into a **new directory**. It does not download models or call a judge.

Use `MODE = "run"` for a new GPU experiment. The model-loading, generation loop, scoring choices, settings, and inspection cells remain visible here. Three small helpers hold reusable mechanics: `eval/checkpoints.py`, `eval/scoring.py`, and `eval/analysis.py`.

References remain Japanese, even for English prompts. They are used for inspection/similarity, not sent to Aya or the guard. Keep dataset text and outputs private.

The expected pair table includes missing and excluded cases. **Complete answers** and **known safety verdicts** are separate conditions; unknown is never silently Safe.


In [ ]:
%pip install -q "auditkit==1.1.3" "pandas>=2.2"


## 1. Get the helper code and choose the experiment

A local checkout is reused. On a fresh Colab runtime this cell clones ez-penta. If the new helpers have not been pushed yet, it asks you to upload the three Python files listed above. Restart the kernel if you change helper code after importing it.

`STAGES = [300]` is a pilot setting for **new** generation. Saved mode inspects the entire supplied export. The original full run has already completed; it need not be generated again.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess

candidates = [Path.cwd(), Path.cwd().parent, Path("/content/ez-penta")]
PROJECT_DIR = next((p for p in candidates if (p / "answercarefully_dataset.py").is_file()), Path("/content/ez-penta"))
if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/Rynhalt/ez-penta.git", str(PROJECT_DIR)], check=True)
helpers = ("checkpoints.py", "scoring.py", "analysis.py")
if any(not (PROJECT_DIR / "eval" / name).is_file() for name in helpers):
    from google.colab import files
    print("Upload eval/checkpoints.py, eval/scoring.py, and eval/analysis.py from your checkout.")
    uploaded = files.upload()
    if not set(helpers) <= uploaded.keys():
        raise ValueError("All three helper files are required.")
    (PROJECT_DIR / "eval").mkdir(parents=True, exist_ok=True)
    for name in helpers:
        (PROJECT_DIR / "eval" / name).write_bytes(uploaded[name])
# Make local helper imports unambiguous when uploading into an older checkout.
for package in ("eval",):
    folder = PROJECT_DIR / package
    folder.mkdir(parents=True, exist_ok=True)
    initializer = folder / "__init__.py"
    if not initializer.exists():
        initializer.write_text('"""Research notebook helpers."""\n')
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))

MODE = "saved"                  # "saved": CPU analysis; "run": new GPU generation + scoring
STAGES = [300]                  # For run mode; [300, None] extends to all complete source rows
RESUME_RUN_ID = ""              # Paste an existing run folder name to resume that configuration
ON_COLAB = Path("/content").exists()
DRIVE_FOLDER = "ez-penta"        # Exact shortcut name shown by this account in mounted My Drive.
SHARED_FOLDER_ID = "1DNqmNqItEl-5ixYcXIS3j-ss_YjPHuib"
FORCE_DRIVE_REMOUNT = False      # Only needed if the mounted directory listing is stale.
DATA_DIR = Path("/content/drive/MyDrive") / DRIVE_FOLDER if ON_COLAB else PROJECT_DIR

AYA_MODEL = "CohereLabs/tiny-aya-water"
GUARD_MODEL = "hf:Qwen/Qwen3Guard-Gen-0.6B"
EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
MAX_NEW_TOKENS = 1024
BATCH_SIZE = 16
MAX_BATCH_TOKENS = 20_000
SCORE_CHUNK = 64
VARIANTS = {
    "original": ("text", "text_en"),
    "fc": ("fc_ja", "fc_en"), "rp": ("rp_ja", "rp_en"), "rs": ("rs_ja", "rs_en"),
    "every10_yes": ("flake_every10_ja_yes", "flake_every10_en_yes"),
    "every10_hai": ("flake_every10_ja_hai", "flake_every10_en_hai"),
    "quartile_yes": ("flake_quartile_ja_yes", "flake_quartile_en_yes"),
    "quartile_hai": ("flake_quartile_ja_hai", "flake_quartile_en_hai"),
}
LANGUAGES = ("ja", "en")
if MODE not in ("saved", "run"):
    raise ValueError("Choose MODE='saved' or MODE='run'.")
if MAX_NEW_TOKENS <= 0 or BATCH_SIZE <= 0 or SCORE_CHUNK <= 0:
    raise ValueError("Token/batch/chunk sizes must be positive.")
if not STAGES or any(n is not None and n <= 0 for n in STAGES):
    raise ValueError("STAGES must contain positive limits or None.")


## 2. Access the shared folder, create a new output directory, and load inputs

**Shared-folder setup (once per Google account):** open the shared folder in Drive and use **Organize → Add shortcut → My Drive**. Set `DRIVE_FOLDER` to the shortcut name visible in the mounted My Drive. Your diagnostic output showed `ez-penta` (hyphen), so that is the default. The original shared-folder title may use `ez_penta` (underscore). This links to the original folder without moving or copying it. A folder URL/ID is not a mounted filesystem path. If the folder is actually in a Workspace Shared drive, set `DATA_DIR` to its mounted `/content/drive/Shareddrives/...` path instead.

The loading cell prints the mount status, configured path, and visible entries containing "penta". Select the Google account where you added the shortcut. If the mount is stale, set `FORCE_DRIVE_REMOUNT = True` once; successful mounting resets it to False. A missing configured shortcut can resolve to the hyphen/underscore alternative only when all three experiment files are present. If needed, use the Colab Files sidebar to copy the exact mounted path into `DATA_DIR`. The notebook checks a known folder-ID target when that implementation-specific location exists, and reports visible names containing "penta" on failure; it does not assume that folder URLs are filesystem paths.

SQLite is read locally for performance; source databases are opened read-only. Saved exports are cross-checked against the generation JSONL, and score fingerprints are checked when results are attached.

All derived results go under **ez-penta/evaluations/<run ID>** through the shared-folder shortcut in Colab, or `evaluations/<run ID>` locally. Completed live responses/scores are checkpointed there. A new run never overwrites the original experiment.

Generation status is `completed / truncated / error / missing`. Safety labels are a separate field: `Safe / Unsafe / Controversial / Unparsed`. An excluded answer has no safety verdict.


In [ ]:
from datetime import datetime, timezone
from importlib.metadata import version
import json
import shutil
import sqlite3
import tempfile
import uuid

import pandas as pd
from IPython.display import display
from eval.checkpoints import ANSWER_KEY, config_id, file_sha256, generation_status, load_jsonl, append_records
from eval.analysis import build_results, load_saved_experiment
from eval.scoring import QWEN3GUARD, JAPANESE_RUN, score_answers

if ON_COLAB and (FORCE_DRIVE_REMOUNT or not Path("/content/drive/MyDrive").is_dir()):
    from google.colab import drive
    drive.mount("/content/drive", force_remount=FORCE_DRIVE_REMOUNT)
    FORCE_DRIVE_REMOUNT = False

if ON_COLAB:
    mounted_root = Path("/content/drive/MyDrive")
    print("My Drive mounted:", mounted_root.is_dir())
    if mounted_root.is_dir():
        matches = [p.name for p in mounted_root.iterdir() if "penta" in p.name.casefold()]
        print("Visible entries containing penta:", matches)
    print("Configured shared-folder path:", DATA_DIR)

if ON_COLAB and not DATA_DIR.is_dir():
    # The mounted shortcut name may differ from the original shared-folder title.
    # Accept only these two names, with the exact experiment files; never the cache folder.
    filenames = ("answercarefully_eval.sqlite3", "eval_aya_answers.jsonl", "eval_scores.jsonl")
    if DATA_DIR.parent == Path("/content/drive/MyDrive") and DATA_DIR.name in ("ez-penta", "ez_penta"):
        for name in ("ez-penta", "ez_penta"):
            candidate = DATA_DIR.parent / name
            if candidate.is_dir() and all((candidate / file).is_file() for file in filenames):
                DATA_DIR = candidate
                print("Using the experiment folder's mounted shortcut name:", DATA_DIR)
                break

if ON_COLAB and not DATA_DIR.is_dir():
    # Some Drive mounts expose shortcut targets by ID instead of the visible name.
    # Check only the supplied folder ID, not a recursive search through personal data.
    target = Path("/content/drive/.shortcut-targets-by-id") / SHARED_FOLDER_ID
    filenames = ("answercarefully_eval.sqlite3", "eval_aya_answers.jsonl", "eval_scores.jsonl")
    for candidate in (target / DRIVE_FOLDER, target):
        if candidate.is_dir() and all((candidate / name).is_file() for name in filenames):
            DATA_DIR = candidate
            print("Using the mounted target of the shared-folder shortcut:", DATA_DIR)
            break
print("Reading experiment files from:", DATA_DIR)
if not DATA_DIR.is_dir():
    my_drive = Path("/content/drive/MyDrive")
    visible = [p.name for p in my_drive.iterdir() if "penta" in p.name.casefold()] if my_drive.is_dir() else []
    raise FileNotFoundError(
        f"Cannot find {DATA_DIR}. MyDrive mounted: {my_drive.is_dir()}; "
        f"visible names containing 'penta': {visible}. "
        "Set FORCE_DRIVE_REMOUNT=True and rerun this cell. Authorize the same Google account "
        "where you added the shortcut. If it is nested or named differently, set DATA_DIR "
        "to its exact mounted path and rerun this cell."
    )
# Derive filenames here so correcting DATA_DIR also corrects every input path.
DB_SAVED = DATA_DIR / "answercarefully_eval.sqlite3"
ANSWERS_SAVED = DATA_DIR / "eval_aya_answers.jsonl"
SCORES_SAVED = DATA_DIR / "eval_scores.jsonl"
DB_PROMPTS = DATA_DIR / "answercarefully_test_prompts_v2.sqlite3"
DB_ORIGINAL = DATA_DIR / "answercarefully.sqlite3"
if MODE == "saved":
    missing = [path.name for path in (DB_SAVED, ANSWERS_SAVED, SCORES_SAVED) if not path.is_file()]
    if missing:
        raise FileNotFoundError(f"Missing in {DATA_DIR}: {missing}. Check DATA_DIR and the shared-folder shortcut.")
RUN_ID = RESUME_RUN_ID or ("evaluation-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "-" + uuid.uuid4().hex[:6])
RUN_DIR = DATA_DIR / "evaluations" / RUN_ID
if RESUME_RUN_ID and not RUN_DIR.is_dir():
    raise FileNotFoundError(f"Cannot resume a run that does not exist: {RUN_DIR}")
RUN_DIR.mkdir(parents=True, exist_ok=bool(RESUME_RUN_ID))
WORK_DIR = Path(tempfile.gettempdir()) / "ez-penta" / RUN_ID
WORK_DIR.mkdir(parents=True, exist_ok=True)
ANSWERS = RUN_DIR / "aya-answers.jsonl"
SCORES = RUN_DIR / "auditkit-scores.jsonl"
DB_OUT = RUN_DIR / "answercarefully_eval.sqlite3"
source_manifest = {"mode": MODE, "variants": {key: list(value) for key, value in VARIANTS.items()}, "helper_hashes": {name: file_sha256(PROJECT_DIR / "eval" / name) for name in helpers}}

def read_records(path, columns="*"):
    connection = sqlite3.connect(path.resolve().as_uri() + "?mode=ro", uri=True)
    try:
        return pd.read_sql_query(f"SELECT {columns} FROM records", connection)
    finally:
        connection.close()

if MODE == "saved":
    required = (DB_SAVED, ANSWERS_SAVED, SCORES_SAVED)
    if any(not p.is_file() for p in required):
        raise FileNotFoundError("Place answercarefully_eval.sqlite3, eval_aya_answers.jsonl, and eval_scores.jsonl in DATA_DIR.")
    local_db = WORK_DIR / "source-evaluation.sqlite3"
    shutil.copyfile(DB_SAVED, local_db)
    answers, all_rows, answer_scores, pair_scores, historical = load_saved_experiment(
        local_db, SCORES_SAVED, ANSWERS_SAVED
    )
    source_manifest["historical_experiment"] = historical
    print(f"Loaded {len(answers):,} saved answers; no inference will run.")
else:
    # Choose historical fallback only after Drive is mounted.
    if not DB_PROMPTS.is_file() and DB_PROMPTS.name == "answercarefully_test_prompts_v2.sqlite3":
        DB_PROMPTS = DATA_DIR / "answercarefully_test_prompts.sqlite3"
    for path in (DB_PROMPTS, DB_ORIGINAL):
        if not path.is_file():
            raise FileNotFoundError(f"Missing {path}; prepare/compile the prompt databases first.")
        shutil.copyfile(path, WORK_DIR / path.name)
    prompts = read_records(WORK_DIR / DB_PROMPTS.name)
    categories = read_records(WORK_DIR / DB_ORIGINAL.name, "source_id, source_split, risk_area_en, harm_type_en")
    prompt_columns = [column for pair in VARIANTS.values() for column in pair]
    complete = prompts[prompt_columns].apply(lambda col: col.map(lambda value: isinstance(value, str) and bool(value.strip()))).all(axis=1)
    quality_columns = [f"{column}_truncated" for column in prompt_columns if f"{column}_truncated" in prompts]
    if quality_columns:
        complete &= ~prompts[quality_columns].fillna(True).astype(bool).any(axis=1)
    all_rows = prompts[complete].merge(categories, on=["source_id", "source_split"], validate="one_to_one", how="left")
    source_manifest.update({"prompt_db_sha256": file_sha256(DB_PROMPTS), "source_db_sha256": file_sha256(DB_ORIGINAL),
                            "rewrite_truncation_columns": quality_columns})
    print(f"Selected {len(all_rows):,}/{len(prompts):,} source rows with all configured prompts.")
    if not quality_columns:
        print("Historical prompt database has no rewrite truncation flags; prompt quality remains unverified.")
    if all_rows.empty:
        raise ValueError("No eligible prompt rows. Inspect the compilation coverage and rewrite completion flags first.")

def rows_for(limit):
    return all_rows if limit is None else all_rows.head(limit)

def jobs_for(rows):
    return pd.DataFrame([
        {"source_id": row.source_id, "source_split": row.source_split, "variant": variant,
         "language": language, "prompt": getattr(row, column), "reference": row.output}
        for row in rows.itertuples(index=False)
        for variant, columns in VARIANTS.items()
        for language, column in zip(LANGUAGES, columns)
    ])

print("Output directory:", RUN_DIR)
print("Source requests:", len(all_rows))
display(all_rows.head(3))


## 3. Live-run setup and Tiny Aya generation

**Saved mode skips this section's work.** Live mode uses the original full-weight float16, greedy generation approach; each request starts a fresh conversation.

Model revisions are resolved before loading and recorded. A resumed run reuses its recorded revisions. Generation checkpoints require a matching generation configuration. Truncated answers remain saved but will not enter individual or pair scoring.

The loop below is visible so you can follow batching, HTTP-free Transformers inference, and checkpointing. GPU memory is released before judging.


In [ ]:
if MODE == "run":
    subprocess.run([sys.executable, "-m", "pip", "install", "transformers>=4.56", "accelerate", "sentencepiece", "protobuf"], check=True)
    import getpass
    from huggingface_hub import HfApi, get_token, login, snapshot_download
    if not get_token():
        token = getpass.getpass("Hugging Face read token (hidden): ").strip()
        try:
            login(token=token, add_to_git_credential=False)
        finally:
            del token
    previous = RUN_DIR / "run-settings.json"
    if previous.is_file():
        previous_settings = json.loads(previous.read_text())
        if previous_settings.get("mode") != "run":
            raise ValueError("A saved-analysis run cannot be resumed as a live experiment.")
        model_revisions = previous_settings["model_revisions"]
    else:
        api = HfApi()
        model_revisions = {
            "aya": api.model_info(AYA_MODEL).sha,
            "guard": api.model_info(GUARD_MODEL.removeprefix("hf:")).sha,
            "embedding": api.model_info(EMBED_MODEL).sha,
        }
    generation_configuration = {
        "protocol": "tiny_aya_chat_v2", "model": AYA_MODEL, "revision": model_revisions["aya"],
        "dtype": "float16", "do_sample": False, "max_new_tokens": MAX_NEW_TOKENS,
        "transformers_version": version("transformers"),
    }
    generation_config_id = config_id(generation_configuration)
    source_manifest.update({"model_revisions": model_revisions, "generation": generation_configuration})
    print("Pinned model revisions:", model_revisions)

manifest_path = RUN_DIR / "run-settings.json"
if manifest_path.exists():
    if json.loads(manifest_path.read_text()) != source_manifest:
        raise ValueError("Run configuration/input hashes changed. Start a new RUN_ID instead of mixing experiments.")
else:
    with manifest_path.open("x") as stream:
        json.dump(source_manifest, stream, ensure_ascii=False, indent=2)


In [ ]:
import gc
if MODE == "run":
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from tqdm.auto import tqdm

def load_aya():
    global aya_model, aya_tokenizer
    if not torch.cuda.is_available():
        raise RuntimeError("No GPU found. In VS Code: Select Kernel -> Colab -> New Colab Server -> GPU (T4).")
    # Rerunning must not load a second copy: two 6.7 GB copies overflow the T4.
    if "aya_model" in globals():
        return
    aya_tokenizer = AutoTokenizer.from_pretrained(AYA_MODEL, revision=model_revisions["aya"], padding_side="left")
    if aya_tokenizer.pad_token_id is None:
        aya_tokenizer.pad_token = aya_tokenizer.eos_token
    # T4s have no fast bfloat16 support, so load in float16.
    aya_model = AutoModelForCausalLM.from_pretrained(AYA_MODEL, revision=model_revisions["aya"], dtype=torch.float16, device_map="cuda")
    aya_model.eval()


def free_aya():
    global aya_model, aya_tokenizer
    if "aya_model" in globals():
        del aya_model, aya_tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def chat_text(prompt):
    return aya_tokenizer.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)


def count_tokens(prompt):
    return len(aya_tokenizer(chat_text(prompt), add_special_tokens=False)["input_ids"])


def generate_batch(prompts):
    """Greedy answers, plus whether each one hit MAX_NEW_TOKENS before finishing."""
    # The chat template already adds BOS, so the tokenizer must not add another.
    encoded = aya_tokenizer([chat_text(prompt) for prompt in prompts], return_tensors="pt", padding=True,
                            add_special_tokens=False).to(aya_model.device)
    with torch.inference_mode():
        output = aya_model.generate(**encoded, max_new_tokens=MAX_NEW_TOKENS, pad_token_id=aya_tokenizer.pad_token_id,
                                    do_sample=False, temperature=None, top_p=None, top_k=None)
    replies = output[:, encoded["input_ids"].shape[1]:]
    eos = aya_model.generation_config.eos_token_id
    stop_ids = set(eos if isinstance(eos, list) else [eos]) | {aya_tokenizer.pad_token_id}
    truncated = [len(reply) == MAX_NEW_TOKENS and int(reply[-1]) not in stop_ids for reply in replies]
    return aya_tokenizer.batch_decode(replies, skip_special_tokens=True), truncated


def make_batches(pending):
    """Pending jobs are sorted by prompt length, so the newest job bounds each batch's memory."""
    batches, current = [], []
    for job in pending:
        too_big = (len(current) + 1) * (job["prompt_tokens"] + MAX_NEW_TOKENS) > MAX_BATCH_TOKENS
        if current and (len(current) == BATCH_SIZE or too_big):
            batches.append(current)
            current = []
        current.append(job)
    if current:
        batches.append(current)
    return batches


def attempt(batch):
    try:
        answers, truncated = generate_batch([job["prompt"] for job in batch])
    except torch.cuda.OutOfMemoryError:
        answers = None
    if answers is None:
        # Freed outside the except block, where the traceback no longer holds the batch's tensors.
        torch.cuda.empty_cache()
        if len(batch) == 1:
            return [(None, False, "OutOfMemoryError: prompt too long for this GPU")]
        middle = len(batch) // 2
        return attempt(batch[:middle]) + attempt(batch[middle:])
    return [(answer.strip(), cut, None) for answer, cut in zip(answers, truncated)]

def valid_answer(record, prompt):
    return bool(record and record.get("prompt") == prompt
                and record.get("config_id") == generation_config_id
                and generation_status(record) in ("completed", "truncated"))

def generate_answers(jobs):
    finished = load_jsonl(ANSWERS, ANSWER_KEY)
    pending = [job for job in jobs.to_dict("records")
               if not valid_answer(finished.get(tuple(job[field] for field in ANSWER_KEY)), job["prompt"])]
    print(f"{len(jobs) - len(pending)}/{len(jobs)} compatible answers; {len(pending)} pending", flush=True)
    try:
        if pending:
            load_aya()
            for job in pending:
                job["prompt_tokens"] = count_tokens(job["prompt"])
            pending.sort(key=lambda job: job["prompt_tokens"])
            for batch in tqdm(make_batches(pending), desc="Tiny Aya"):
                records = []
                for job, (answer, truncated, error) in zip(batch, attempt(batch)):
                    record = {**{field: job[field] for field in ANSWER_KEY}, "prompt": job["prompt"],
                              "model": AYA_MODEL, "model_revision": model_revisions["aya"],
                              "max_new_tokens": MAX_NEW_TOKENS, "config_id": generation_config_id,
                              "answer": answer, "truncated": truncated, "error": error}
                    record["generation_status"] = generation_status(record)
                    records.append(record)
                append_records(ANSWERS, records)
    finally:
        free_aya()
    finished = load_jsonl(ANSWERS, ANSWER_KEY)
    output = jobs.copy()
    found = [finished.get(tuple(row[field] for field in ANSWER_KEY)) for row in jobs.to_dict("records")]
    found = [record if valid_answer(record, prompt) else {} for record, prompt in zip(found, jobs["prompt"])]
    for field in ("answer", "truncated", "error"):
        output[field] = [record.get(field) for record in found]
    return output


## 4. Choose the AuditKIT measurements

**Saved mode reuses historical judgments and does not load a judge.** Imported scores retain unknown historical provenance and cannot silently become a new live cache.

For live runs, these are the measurements:
- `GuardJudge` classifies the answer in the context of its question; a separate scorer classifies the incoming prompt.
- `ChrF` and multilingual cosine similarity compare text, not safety. ChrF is reported only for Japanese references.
- Japanese-script detection is only a heuristic.
- Pair similarity is measured only when **both answers completed**.

`score_answers` creates `ak.Sample` objects, calls `ak.evaluate(..., model="precomputed")`, and checkpoints the resulting scores. Its cache identity includes judge configuration, metric/annotator identities, parser code, and AuditKIT version. Failed scoring is kept separate from an uncertain verdict and is retried on resume.


In [ ]:
if MODE == "run":
    import auditkit as ak
    from auditkit.annotator import RegexAnnotator

    embedding_path = snapshot_download(EMBED_MODEL, revision=model_revisions["embedding"])
    guard_backend = ak.AutoModel.resolve(GUARD_MODEL, revision=model_revisions["guard"])
    ANSWER_SCORERS = [
        ak.GuardJudge(QWEN3GUARD, judge_model=guard_backend, name="guard_response"),
        ak.GuardJudge(QWEN3GUARD, judge_model=guard_backend, assess="prompt", name="guard_prompt"),
        ak.ChrF(),
        ak.CosineSimilarity(model_name=embedding_path),
    ]
    ANSWER_ANNOTATORS = [RegexAnnotator(pattern=JAPANESE_RUN, name="japanese_text")]
    PAIR_SCORERS = [ak.CosineSimilarity(model_name=embedding_path)]
    score_configuration = {"guard_model": GUARD_MODEL, "guard_revision": model_revisions["guard"],
                           "embedding_model": EMBED_MODEL, "embedding_revision": model_revisions["embedding"],
                           "guard_temperature": 0.0, "guard_max_tokens": 128, "protocol": "paired_safety_v2"}


## 5. Build comparisons and save results

The expected source × variant × language index is built **before** attaching observations. Missing languages, truncated generations, execution failures, unknown verdicts, and missing judgments remain inspectable.

Unsafe rate uses only binary Safe/Unsafe labels. Refusal has its own known-label denominator. Every rate includes its denominator; raw values retain precision. Exact paired tests describe guard outcomes, not validated human judgments. Adjusted language-comparison p-values cover two outcomes per configured variant.

The saved `pairs` table keeps every expected pair. `transitions` compares originals and variants on matched known observations. `risk_counts` provides denominators behind the risk-area view.


In [ ]:
from contextlib import closing

def save_results(results):
    local = WORK_DIR / "derived-evaluation.sqlite3"
    local.unlink(missing_ok=True)
    names = {
        "table": "answers", "pairs": "pairs",
        "by_variant_language": "summary_by_variant_language",
        "change_vs_original": "summary_change_vs_original",
        "by_variant": "summary_by_variant", "by_risk_area": "summary_refusal_mismatch_by_risk_area",
        "risk_counts": "risk_counts", "transitions": "transitions",
    }
    with closing(sqlite3.connect(local)) as connection:
        for name, sql_name in names.items():
            frame = results[name]
            if name in ("by_variant_language", "change_vs_original", "by_variant", "by_risk_area"):
                frame = frame.reset_index()
            frame.to_sql(sql_name, connection, index=False)
            if name != "table":  # Full answers already live in SQLite; avoid a duplicate text dump.
                frame.to_csv(RUN_DIR / f"{sql_name}.csv", index=False)
        info = {"run_id": RUN_ID, "mode": MODE, "analysis_protocol": "paired_safety_v2",
                "source_requests": len(results["pairs"][["source_id", "source_split"]].drop_duplicates()),
                "variants": ", ".join(VARIANTS), "auditkit_version": version("auditkit"),
                "source_manifest": json.dumps(source_manifest, ensure_ascii=False)}
        pd.DataFrame([info]).to_sql("eval_info", connection, index=False)
        connection.commit()
    pending = RUN_DIR / "derived-evaluation.partial"
    shutil.copyfile(local, pending)
    pending.replace(DB_OUT)
    if MODE == "run" and len(STAGES) > 1:
        snapshot = RUN_DIR / f"answercarefully_eval_{info['source_requests']}rows.sqlite3"
        shutil.copyfile(local, snapshot)
    print("Saved new results:", DB_OUT, flush=True)

if MODE == "saved":
    results = build_results(answers, all_rows, VARIANTS, answer_scores, pair_scores)
    save_results(results)
else:
    for limit in STAGES:
        rows = rows_for(limit)
        print(f"Stage: {len(rows)} source rows x {len(VARIANTS)} variants x 2 languages")
        answers = generate_answers(jobs_for(rows))
        try:
            answer_scores, pair_scores = score_answers(
                answers, VARIANTS, ANSWER_SCORERS, PAIR_SCORERS, SCORES,
                score_configuration, ANSWER_ANNOTATORS, SCORE_CHUNK
            )
        finally:
            guard_backend.unload()  # Do not retain guard weights during the next Aya stage.
        results = build_results(answers, rows, VARIANTS, answer_scores, pair_scores)
        save_results(results)


## 6. Inspect coverage before interpreting rates

For the supplied full baseline, the expected checks are **14,400 pairs**, **13,913 complete pairs**, and **13,051 pairs with binary safety judgments in both languages**. The corrected similarity count is 13,913, not 14,400.

These numbers describe different eligibility conditions. An uncertain guard label is not an inference failure and is not Safe.


In [ ]:
pairs = results["pairs"]
coverage = pd.Series({
    "expected_answers": len(results["table"]),
    "expected_pairs": len(pairs),
    "complete_answer_pairs": int(pairs["answers_complete"].sum()),
    "binary_safety_pairs": int(pairs["safety_known"].sum()),
    "known_refusal_pairs": int(pairs["refusal_known"].sum()),
    "similarity_pairs": int(pairs["similarity_ja_en"].notna().sum()),
})
display(coverage.to_frame("count"))
display(results["by_variant_language"].round(4))


## 7. Compare behavior and inspect individual cases

A changed safety/refusal decision is a candidate inconsistency. Review the actual prompts and answers to separate Aya behavior, evaluator error, and changed prompt meaning. Refusal alone cannot establish over-refusal.

`by_variant` compares languages; `transitions` compares each variant against the original within a language. The original-to-variant aggregate-rate differences are descriptive and may use different cohorts; use matched transitions for source-level conclusions.


In [ ]:
display(results["by_variant"].round(4))
display(results["transitions"])
display(results["risk_counts"].round(4))
display(results["change_vs_original"].round(4))


In [ ]:
VARIANT_TO_INSPECT = "original"
SOURCE_ID_TO_INSPECT = ""  # Paste an exact source ID; empty selects the first safety disagreement.
candidates = pairs[pairs["variant"].eq(VARIANT_TO_INSPECT) & pairs["safety_known"]
                   & pairs["unsafe_ja"].ne(pairs["unsafe_en"])]
source_id = SOURCE_ID_TO_INSPECT or (candidates.iloc[0]["source_id"] if len(candidates) else None)
if source_id:
    pd.set_option("display.max_colwidth", None)
    selected = results["table"][results["table"]["source_id"].eq(source_id)
               & results["table"]["variant"].isin(("original", VARIANT_TO_INSPECT))]
    display(selected[["source_id", "source_split", "variant", "language", "prompt", "answer",
                      "generation_status", "safety", "refused", "scoring_status", "exclusion_reason"]])
else:
    print("No binary safety disagreement in this selection. Set SOURCE_ID_TO_INSPECT to inspect another case.")
print("Results are in:", RUN_DIR)


## Next experiment

Keep this completed run unchanged. For a new configuration, clear `RESUME_RUN_ID` and rerun from section 1. To continue an interrupted live run, set its folder name in `RESUME_RUN_ID` and keep the same inputs/settings. Malformed checkpoint lines stop with their location rather than silently discarding records.

Reanalysis can correct filtering and denominators, but cannot establish prompt equivalence or judge accuracy. Human review remains necessary. No publication or extra toolkit integration is performed here.
